In [3]:
import pandas as pd
import spacy
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import RandomOverSampler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV

# Load your CSV file
data = pd.read_csv("supply_chain_project_trustpilot_advanced_merge.csv")

# Orview
data.head()


,Unnamed: 0,Company,Name,Rating_number_customer,Heading,Comment,Stars,Invitation,Dates
0,0,skatedeluxe,Sandra,2,Jederzeit wieder,"<p class=""typography_body-l__v5JLj typography_...",5,Auf Einladung,5. März 2025
1,1,skatedeluxe,customer,2,Schnelle Lieferung,No comment,5,Auf Einladung,5. März 2025
2,2,skatedeluxe,Dexter,1,Bester Service und top Qualität,"<p class=""typography_body-l__v5JLj typography_...",5,Auf Einladung,4. März 2025
3,3,skatedeluxe,Stephan Lameck,1,Schnelligkeit,"<p class=""typography_body-l__v5JLj typography_...",5,Auf Einladung,4. März 2025
4,4,skatedeluxe,Fritz Brack,2,Super Service,"<p class=""typography_body-l__v5JLj typography_...",5,Auf Einladung,3. März 2025


In [4]:
# Clean up HTML-like text in comments
def extract_comment(text):
    if isinstance(text, str):
        matches = re.findall(r'>([^<]+)<', text)
        if matches:
            return matches[0]
    return text  # If no string or no match, return the text unchanged

# Apply the clean-up
data['Comment'] = data['Comment'].apply(extract_comment)

# Merge headline and comment
data['Text'] = data['Heading'].fillna('') + ' ' + data['Comment'].fillna('')

# Remove unused columns
data = data.drop(['Name', 'Heading', 'Comment'], axis=1)

# Orview
data.head()

,Unnamed: 0,Company,Rating_number_customer,Stars,Invitation,Dates,Text
0,0,skatedeluxe,2,5,Auf Einladung,5. März 2025,"Jederzeit wieder Sehr schnelle Lieferung, gute..."
1,1,skatedeluxe,2,5,Auf Einladung,5. März 2025,Schnelle Lieferung No comment
2,2,skatedeluxe,1,5,Auf Einladung,4. März 2025,Bester Service und top Qualität Der bestellvor...
3,3,skatedeluxe,1,5,Auf Einladung,4. März 2025,Schnelligkeit Ausgefallene Produkte
4,4,skatedeluxe,2,5,Auf Einladung,3. März 2025,"Super Service Super Service, extrem schnelle L..."


In [5]:
# Load the German language model
nlp = spacy.load('de_core_news_sm')

# Tokeniser + Lemmatiser combined
def preprocess_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return tokens

# Add tokenised and lemmatised text
data['tokens'] = data['Text'].apply(preprocess_text)

# Only for TF-IDF later, convert back to space-separated text
data['clean_text'] = data['tokens'].apply(lambda x: ' '.join(x))


In [6]:
# Orview
data.head()


,Unnamed: 0,Company,Rating_number_customer,Stars,Invitation,Dates,Text,tokens,clean_text
0,0,skatedeluxe,2,5,Auf Einladung,5. März 2025,"Jederzeit wieder Sehr schnelle Lieferung, gute...","[Jederzeit, schnell, Lieferung, Preis-Leistung...",Jederzeit schnell Lieferung Preis-Leistungsver...
1,1,skatedeluxe,2,5,Auf Einladung,5. März 2025,Schnelle Lieferung No comment,"[schnell, Lieferung, --, comment]",schnell Lieferung -- comment
2,2,skatedeluxe,1,5,Auf Einladung,4. März 2025,Bester Service und top Qualität Der bestellvor...,"[Bester, Service, Top, Qualität, Bestellvorgan...",Bester Service Top Qualität Bestellvorgang unk...
3,3,skatedeluxe,1,5,Auf Einladung,4. März 2025,Schnelligkeit Ausgefallene Produkte,"[Schnelligkeit, Ausgefallene, Produkt]",Schnelligkeit Ausgefallene Produkt
4,4,skatedeluxe,2,5,Auf Einladung,3. März 2025,"Super Service Super Service, extrem schnelle L...","[Super, Service, Super, Service, extrem, schne...",Super Service Super Service extrem schnell Lie...


In [7]:
# Separate features and labels
X = data['clean_text']
y = data['Stars']

# Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectoriser
vec = TfidfVectorizer()

# Adapt to training, transform both
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

# oversampling
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train_vec, y_train)

# Modeltesting: Which model has the best accuracy?

models = {
    'Gradient Boosting': GradientBoostingClassifier(),
    'Random Forest': RandomForestClassifier(),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier()
}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    print(f'{name} Accuracy: {accuracy_score(y_test, y_pred)}')
    print(classification_report(y_test, y_pred))

Gradient Boosting Accuracy: 0.7821001088139282
              precision    recall  f1-score   support

           1       0.74      0.81      0.77       966
           2       0.85      0.07      0.13       159
           3       0.72      0.12      0.21       192
           4       0.50      0.04      0.07       237
           5       0.80      0.97      0.88      2122

    accuracy                           0.78      3676
   macro avg       0.72      0.40      0.41      3676
weighted avg       0.76      0.78      0.73      3676

Random Forest Accuracy: 0.8658868335146899
              precision    recall  f1-score   support

           1       0.82      0.93      0.87       966
           2       0.99      0.47      0.64       159
           3       0.95      0.47      0.63       192
           4       0.93      0.27      0.41       237
           5       0.88      0.97      0.92      2122

    accuracy                           0.87      3676
   macro avg       0.91      0.62      0.

In [8]:
# RandomForest deliver nearly the same accuracy of 0.86!

In [9]:
# Grid Search applied to Rondom Forest
# Model
model_rf = RandomForestClassifier(random_state=42)

# Parametergrid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20]
}

# GridSearch
grid_search_rf = GridSearchCV(estimator=model_rf, param_grid=param_grid,
                           scoring='accuracy', cv=5, n_jobs=-1, verbose=1)

# Train on resampled data!
grid_search_rf.fit(X_resampled, y_resampled)


# Beste Parameter + Score
print("Beste Parameter:", grid_search_rf.best_params_)
print("Beste Accuracy:", grid_search_rf.best_score_)

best_rf = grid_search_rf.best_estimator_
y_pred = best_rf.predict(X_test_vec)

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Fitting 5 folds for each of 6 candidates, totalling 30 fits
Beste Parameter: {'max_depth': None, 'n_estimators': 200}
Beste Accuracy: 0.9821211760504701
Test Accuracy: 0.8558215451577802
              precision    recall  f1-score   support

           1       0.83      0.93      0.88       966
           2       0.93      0.49      0.64       159
           3       0.82      0.48      0.61       192
           4       0.50      0.32      0.39       237
           5       0.89      0.94      0.92      2122

    accuracy                           0.86      3676
   macro avg       0.79      0.63      0.69      3676
weighted avg       0.85      0.86      0.84      3676



In [10]:
# Grid Search applied to Logistic Regression
# Model
model_logreg = LogisticRegression(max_iter=1000, random_state=42)

# Parametergrid
param_grid = {
    'C': [0.1, 1, 10],          # Regularisation (lower value -> more regularisation
    'penalty': ['l2'],          # L2 regularisation (there is also ‘l1’, but ‘l2’ is used more frequently)
    'solver': ['lbfgs', 'liblinear'],  # Solvers used in logistic regressions
}

# GridSearch
grid_search_logreg = GridSearchCV(estimator=model_logreg, param_grid=param_grid,
                           scoring='accuracy', cv=5, n_jobs=-1, verbose=1)

# Train on resampled data!
grid_search_logreg.fit(X_resampled, y_resampled)


# Beste Parameter + Score
print("Beste Parameter:", grid_search_logreg.best_params_)
print("Beste Accuracy:", grid_search_logreg.best_score_)

best_logreg = grid_search_logreg.best_estimator_
y_pred_logreg = best_logreg.predict(X_test_vec)

print("Test Accuracy:", accuracy_score(y_test, y_pred_logreg))
print(classification_report(y_test, y_pred_logreg))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Beste Parameter: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
Beste Accuracy: 0.9310796333769789
Test Accuracy: 0.8161044613710555
              precision    recall  f1-score   support

           1       0.87      0.89      0.88       966
           2       0.60      0.60      0.60       159
           3       0.53      0.61      0.57       192
           4       0.31      0.43      0.36       237
           5       0.92      0.86      0.89      2122

    accuracy                           0.82      3676
   macro avg       0.65      0.68      0.66      3676
weighted avg       0.84      0.82      0.82      3676



In [11]:
# Grid Search applied to SVC (Support Vector Classifier)
# Model
model_svm = SVC(random_state=42)


# Parametergrid
param_grid = {
    'C': [0.1, 1, 10],           # Regularisierung: kleiner C-Wert -> mehr Regularisierung
    'kernel': ['linear', 'rbf'], # Kernel-Auswahl: 'linear' oder 'rbf' (radial basis function)
    'gamma': ['scale', 'auto']   # gamma steuert die Form der Entscheidungsgrenze
}

# GridSearch
grid_search_svm = GridSearchCV(estimator=model_svm, param_grid=param_grid,
                           scoring='accuracy', cv=5, n_jobs=-1, verbose=1)

# Train on resampled data!
grid_search_svm.fit(X_resampled, y_resampled)


# Beste Parameter + Score
print("Beste Parameter:", grid_search_svm.best_params_)
print("Beste Accuracy:", grid_search_svm.best_score_)

best_svm = grid_search_svm.best_estimator_
y_pred_svm = best_svm.predict(X_test_vec)

print("Test Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Beste Parameter: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
Beste Accuracy: 0.9828353767408643
Test Accuracy: 0.8634385201305768
              precision    recall  f1-score   support

           1       0.86      0.95      0.90       966
           2       0.95      0.48      0.64       159
           3       0.78      0.54      0.64       192
           4       0.45      0.33      0.38       237
           5       0.90      0.94      0.92      2122

    accuracy                           0.86      3676
   macro avg       0.79      0.65      0.70      3676
weighted avg       0.86      0.86      0.85      3676



In [17]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# Assuming you already have your true labels (y_test) and the predictions (y_pred) 
# multiclass classification with ratings 1 to 5

# Calculate the Confusion Matrix:
cm = confusion_matrix(y_test, y_pred)

# Set the classes (e.g. if your ratings are 1-5 stars)
labels = [1, 2, 3, 4, 5]

# Create a DataFrame from the Confusion Matrix
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

# Save the Confusion Matrix as a CSV file
cm_df.to_csv("confusion_matrix.csv", index=True)

# Output of the confusion matrix as a DataFrame
print("Confusion Matrix (als DataFrame):")
print(cm_df)

Confusion Matrix (als DataFrame):
     1   2   3   4     5
1  902   1   3  19    41
2   55  78   1   3    22
3   42   3  93   7    47
4   28   1   5  76   127
5   65   1  11  48  1997


In [19]:
print("Gesamtzahl der Zeilen:", data.shape[0])

Gesamtzahl der Zeilen: 18378
